
####Objetivo del notebook

1. Calcular el centroide del grupo muerte.
2. Hallar el porcentaje de representatividad de los casos en relación a la categoría de centro o centroide. Esta medida se analizará en el notebook 03_5

-Entrada:

Base de datos: nofatales_muerte_var_significativas , cuyo origen es notebook 01_7

-Salida:

Tabla: representatividad_muerte

In [0]:
#verificar si kmodes forma parte de las bibliotecas estándar de Databricks
try:
    from kmodes.kmodes import KModes
    print("La librería kmodes está instalada.")
except ImportError:
    print("La librería kmodes NO está instalada.")

In [0]:
%pip install kmodes

In [0]:
%restart_python

In [0]:
#Verificar la instalación

from kmodes.kmodes import KModes

print(KModes)

In [0]:

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat
from pyspark.sql.window import Window
from functools import reduce

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# K-Modes
from kmodes.kmodes import KModes

# Tiempo de ejecución
import time

# Ignorar advertencias
import warnings
warnings.filterwarnings("ignore")

In [0]:
#Leer la tabla base
df_base = spark.table(
    "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"
)

In [0]:
df_nofatales = df_base.filter(col("grupo")=="nofatales")
df_muerte = df_base.filter(col("grupo")=="muerte")

In [0]:
print("No fatales:", df_nofatales.count())

print("Muerte:", df_muerte.count())

In [0]:
variables_kmodes = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

In [0]:
def obtener_centroide_moda(df, variables):

    centroide = {}

    for variable in variables:

        moda = (
            df.groupBy(variable)
              .count()
              .orderBy(col("count").desc())
              .first()[0]
        )

        centroide[variable] = moda

    return centroide

In [0]:
centroide_muerte = obtener_centroide_moda(
    df_muerte,
    variables_kmodes
)

centroide_muerte

In [0]:
centroide_muerte ={
    'sexo_victima_cod': 'Hombre',
    'ciclo_vital_cod': '(12 a 17) Adolescencia',
    'escolaridad_cod': 'Primaria',
    'estado_civil_cod': 'Soltero (a)',
    'contexto_del_hecho_cod': 'Lesiones no Fatales contra Niños, Niñas y Adolescentes por Violencia Intrafamiliar',
    'mecanismo_causal_cod': 'Contundente',
    'presunto_agresor_cod': 'Padre-Padrastro'
}

In [0]:
import pandas as pd

fila_muerte = {
    "cluster": -1,
    **centroide_muerte
}

df_centroide_muerte = pd.DataFrame([fila_muerte])

df_centroide_muerte

In [0]:
df_centroide_muerte_spark = spark.createDataFrame(df_centroide_muerte)

In [0]:
df_centroide_muerte_spark.printSchema()

In [0]:
(
    df_centroide_muerte_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.centroide_muerte"
    )
)

In [0]:
spark.table(
    "ml_proyecto_7405607705157039.default.centroide_muerte"
).display()

**Tabla del porcentaje que representa cada categoría modal dentro de su cluster**

In [0]:
#variables de estudio

variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

In [0]:
resultados = []

total = df_muerte.count()

for variable in variables:

    moda = (
        df_muerte
        .groupBy(variable)
        .count()
        .withColumn(
            "porcentaje",
            F.round(
                100 * F.col("count") / total,
                2
            )
        )
    )

    ventana = Window.orderBy(F.desc("count"))

    moda = (
        moda
        .withColumn(
            "rn",
            F.row_number().over(ventana)
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    moda = (
        moda
        .withColumn("variable", F.lit(variable))
        .withColumnRenamed(variable, "categoria_modal")
        .withColumnRenamed("count", "frecuencia")
        .withColumn("total", F.lit(total))
    )

    resultados.append(moda)

In [0]:
df_representatividad_muerte = reduce(
    lambda x, y: x.unionByName(y),
    resultados
)

In [0]:
df_representatividad_muerte = df_representatividad_muerte.select(
    "variable",
    "categoria_modal",
    "frecuencia",
    "total",
    "porcentaje"
)

display(df_representatividad_muerte)

In [0]:
(
    df_representatividad_muerte
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.representatividad_muerte"
    )
)

In [0]:
spark.table(
    "ml_proyecto_7405607705157039.default.representatividad_muerte"
).display()